In [15]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [16]:
import os
import kagglehub
import pickle
import numpy as np
import librosa
from scipy.stats import kurtosis, skew

In [17]:
# Configuration
SAMPLE_RATE = 16000
CLIP_DURATION = 3
TARGET_LEN = SAMPLE_RATE * CLIP_DURATION
N_MFCC = 13
N_MELS = 40
N_MFCC_QUARTERS = 4

In [18]:
path1 = kagglehub.dataset_download("vbookshelf/respiratory-sound-database")
path2 = kagglehub.dataset_download("tareqkhanemu/snoring")
print("ICBHI path:", path1)
print("Snoring path:", path2)

ICBHI_PATH = os.path.join(
    path1,
    "Respiratory_Sound_Database",
    "Respiratory_Sound_Database",
    "audio_and_txt_files"
)
SNORING_PATH = os.path.join(path2, "Snoring Dataset")

wav_count = len([f for f in os.listdir(ICBHI_PATH)   if f.endswith('.wav')])
snore_count = len([f for f in os.listdir(os.path.join(SNORING_PATH, '1')) if f.endswith('.wav')])
print(f"\nICBHI wavs: {wav_count}")
print(f"Snore clips: {snore_count}")

ICBHI path: /kaggle/input/datasets/vbookshelf/respiratory-sound-database
Snoring path: /kaggle/input/datasets/tareqkhanemu/snoring

ICBHI wavs: 920
Snore clips: 500


In [19]:
# audio file for testing
TEST_AUDIO_PATH = "/kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/101_1b1_Al_sc_Meditron.wav"

In [20]:
# Load exactly 3 seconds to get our 48,000 samples
test_audio, _ = librosa.load(TEST_AUDIO_PATH, sr=SAMPLE_RATE, duration=CLIP_DURATION, mono=True)

In [21]:
# Pad with zeros if the clip is somehow shorter than 3 seconds
if len(test_audio) < TARGET_LEN:
    test_audio = np.pad(test_audio, (0, TARGET_LEN - len(test_audio)))

In [22]:
def extract_features(audio, sr=SAMPLE_RATE):
    feats = []
    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc  = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size   = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]          
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(    mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

In [23]:
# Define the path to your file
file_path1 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/encoder_v3.pkl'
file_path2 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/ensemble_v3.pkl'
file_path3 = '/kaggle/input/datasets/shafayatulislam/trainedmodels/scaler_v3.pkl'

In [24]:
# Load Models
with open(file_path3, 'rb') as f:
    scaler = pickle.load(f)
with open(file_path1, 'rb') as f:
    encoder = pickle.load(f)
with open(file_path2, 'rb') as f:
    ensemble = pickle.load(f)

In [25]:
# Extract and Normalize
raw_features = extract_features(test_audio)
normalized_features = scaler.transform(raw_features.reshape(1, -1))[0]

/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


In [26]:
# Inference using Voting Ensemble
probabilities = ensemble.predict_proba(normalized_features.reshape(1, -1))[0]
max_index = np.argmax(probabilities)
max_confidence = probabilities[max_index]
predicted_label = encoder.classes_[max_index]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [27]:
# Print the result
for i, class_name in enumerate(encoder.classes_):
    prob_percentage = probabilities[i] * 100
    print(f"{class_name.capitalize():<8} : {prob_percentage:>5.1f}%")

Crackle  :   5.3%
Normal   :  63.8%
Snore    :   0.0%
Wheeze   :  30.9%


In [28]:
if max_confidence > 0.75:
    print(f"[DETECTED] {predicted_label.capitalize()} ({max_confidence * 100:.1f}%)")
else:
    print(f"[DISCARDED] Below 75% threshold (Highest: {predicted_label.capitalize()} at {max_confidence * 100:.1f}%)")

[DISCARDED] Below 75% threshold (Highest: Normal at 63.8%)
